In [121]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import os
import io
import builtins
import time
import re
from itertools import product
import dash
from dash import dcc
from dash import html 
from dash import dash_table
import dash_bootstrap_components as dbc


from dash.dependencies import Input, Output
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 63)

In [122]:
python_exports = "C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports"
current_save = "Leverkusen"
all_players_file = "all_players_data.csv"

In [123]:
folder_path = python_exports + "/" + current_save
all_players_file_path = folder_path + "/" + all_players_file
all_folder_files = os.listdir(folder_path)


In [124]:
all_html_files = [f for f in all_folder_files if f.endswith('.html')]    
all_html_files = [w.replace('.html', '') + ' 00:00:01' for w in all_html_files]


In [125]:
if not os.path.exists(all_players_file_path):
    exported_dates = pd.DataFrame({'Date': [1, 2]})

else:
    # If the file exists, read the existing data and make a copy    
    exported_dates = pd.read_csv(all_players_file_path)

exported_dates = exported_dates.Date.unique().tolist()


In [126]:
for file in all_html_files:
    if file not in exported_dates:
        file_name = file.replace(' 00:00:01','')
        full_path = folder_path + '/' + file_name + '.html'
        fm_export= bs(open(full_path,encoding = "utf8"), "html.parser").find('table')
        raw_list = pd.read_html(io.StringIO(str(fm_export)))
        raw_table = pd.DataFrame(raw_list[0])
        date = file_name.split('.')[0] +  ' 00:00:01'
        raw_table['Date'] = pd.to_datetime(date, format = "%Y-%m-%d %H:%M:%S")
        raw_table['DoB'] = pd.to_datetime(raw_table["DoB"].str.split(" ").str[0], format="%d/%m/%Y")
        raw_table['DoB'] = raw_table['DoB'] + pd.Timedelta(seconds=1)
        raw_table['Age'] = (raw_table['Date']-raw_table['DoB']).dt.days/365.25
        raw_table['Acc'] = pd.to_numeric(raw_table['Acc'], errors='coerce')

        raw_table = raw_table.dropna(subset=['Acc'])
        if not os.path.exists(all_players_file_path):
            raw_table.to_csv(all_players_file_path, index=False)
            print(f"Created file: {all_players_file_path}")
        else:
            # If the file exists, read the existing data and make a copy
            timestr = time.strftime("%Y%m%d-%H%M%S")
            existing_df = pd.read_csv(all_players_file_path)
            csv_filename = f'C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/{current_save}/all_players_data_{timestr}.csv'
            existing_df.to_csv(csv_filename, index=False)

            # Union the existing DataFrame with the new DataFrame
            combined_df = pd.concat([existing_df, raw_table]).drop_duplicates().reset_index(drop=True)

            # Save the combined DataFrame back to the CSV file
            combined_df.to_csv(all_players_file_path, index=False)
            print(f"Updated file: {all_players_file_path}")


In [127]:
full_path = folder_path + '/' + 'all_players_data' + '.csv'

In [128]:
raw_table = pd.read_csv(full_path)


In [129]:
def process_segment(segment):
    segment = segment.replace("D/","Z/").replace("D ",("Z "))
    sides = re.findall(r'\((.*?)\)', segment)
    sides = sides[0]    
    lines = segment.replace("("+sides+")","").replace("/","").replace("AM","A").replace("DM","O").replace('ST','F').replace("WB","O").replace("Z","D").strip()
    result = [f"{x}{y}" for x, y in product(lines, sides)] 
    print("result")
    print(result)
        
        
    return result

In [130]:
def combine_segment(segments):
    result = []
    segments = segments.replace('GK','G (C)')
    segments = re.sub(r'\bDM\b(,?)', r'DM (C)\1', segments)
    for segment in segments.split(','):
        result.extend(process_segment(segment.strip()))
    print("combine segment")    
    print(result)
    return result


In [131]:
def apply_combinations(df, column):
    
    df["Positions"] = df[column].apply(combine_segment)
    df['Positions'] = df['Positions'].apply(replace_values)
    print("apply comb")
    print(df)
    return df


In [132]:
def replace_values(lst):
    new_lst = []
    for item in lst:
        item = item.replace('O', 'DM')
        item = item.replace('GC', 'GK')
        item = item.replace('FC', 'ST')
        item = item.replace('A', 'AM')
        item = item.replace('DMR', 'WBR')
        item = item.replace('DML', 'WBL')
        item = item.replace('DMC', 'DM')


        new_lst.append(item)
        print("replace")
        print(new_lst)
    return new_lst

In [133]:
#raw_table = raw_table[raw_table['Name']=='Edmond Tapsoba']
df = apply_combinations(raw_table, 'Position')


result
['MC']
result
['AR', 'AC']
combine segment
['MC', 'AR', 'AC']
result
['AL']
combine segment
['AL']
result
['DR', 'DL']
result
['OR', 'MR', 'AR']
combine segment
['DR', 'DL', 'OR', 'MR', 'AR']
result
['DR', 'DC']
combine segment
['DR', 'DC']
result
['DC']
combine segment
['DC']
result
['DR']
result
['OR', 'OL']
result
['MR']
combine segment
['DR', 'OR', 'OL', 'MR']
result
['DR', 'DC']
result
['OR']
combine segment
['DR', 'DC', 'OR']
result
['AC']
result
['FC']
combine segment
['AC', 'FC']
result
['DL', 'DC']
result
['OC']
combine segment
['DL', 'DC', 'OC']
result
['GC']
combine segment
['GC']
result
['DR', 'DC']
result
['OC']
combine segment
['DR', 'DC', 'OC']
result
['DC']
result
['OC']
combine segment
['DC', 'OC']
result
['MR']
result
['AR', 'AL']
result
['FC']
combine segment
['MR', 'AR', 'AL', 'FC']
result
['MR', 'MC']
result
['AR']
result
['FC']
combine segment
['MR', 'MC', 'AR', 'FC']
result
['DC']
result
['OC']
combine segment
['DC', 'OC']
result
['ML']
result
['AL', 'AC']

In [134]:
unique_values = set([item for sublist in df['Positions'] for item in sublist])
print(unique_values)

{'MC', 'GK', 'DC', 'AML', 'AMR', 'DR', 'MR', 'DL', 'AMC', 'ST', 'ML', 'WBL', 'DM', 'WBR'}


In [135]:
for position in unique_values:
    df[position] = df.apply(lambda x: 1 if position in x["Positions"] else 0, axis=1)
print(df)

               UID  Inf                  Name             Position  Nat  \
0     r-2002137010  Yth             Ady Heale       M (C), AM (RC)  ENG   
1     r-2002137003  Wnt         Gaku Ideguchi               AM (L)  GER   
2     r-2002111705  Ctr              Kadir Ok  D (RL), WB/M/AM (R)  TUR   
3         93127195  Wnt      Odilon Kossounou               D (RC)  CIV   
4     r-2002111746  Wnt   Julian Hutton-Woods                D (C)  ENG   
...            ...  ...                   ...                  ...  ...   
1740    2000104892  NaN    Eliesse Ben Seghir       M (C), AM (RC)  FRA   
1741  r-2002122517  Inj  Liam Van Den Bossche      AM (RL), ST (C)  BEL   
1742  r-2002120802  Sct  Giorgos Konstantinou       D (RL), WB (R)  CYP   
1743  r-2002069741  Sct       Matthias Weigel             D/WB (R)  GER   
1744      18115012  NaN           Roméo Lavia            DM, M (C)  BEL   

            Age                  DoB                 Club           Division  \
0     15.838467  20

In [136]:
df

,UID,Inf,Name,Position,Nat,Age,DoB,Club,Division,Transfer Value,Personality,Media Handling,Left Foot,Right Foot,Pac,Acc,Jum,1v1,Aer,Agg,Agi,Ant,Bal,Bra,Cmd,Cnt,Cmp,Cro,Dec,Det,Dri,...,Tck,Tea,Tec,Thr,TRO,Vis,Wor,Cor,Com,Ecc,Fre,L Th,Nat.1,Pen,Pun,Date,Positions,MC,GK,DC,AML,AMR,DR,MR,DL,AMC,ST,ML,WBL,DM,WBR
0,r-2002137010,Yth,Ady Heale,"M (C), AM (RC)",ENG,15.838467,2013-08-29 00:00:01,Bayer Leverkusen,Bundesliga,£12K - £120K,Balanced,Outspoken,Strong,Very Strong,11,8,7,2,3,11,11,5,10,5,3,6,8,5,8,9,5,...,5,8,13,2,2,11,8,6,3,3,4,1,11,4,2,2029-07-01 00:00:01,"[MC, AMR, AMC]",1,0,0,0,1,0,0,0,1,0,0,0,0,0
1,r-2002137003,Wnt,Gaku Ideguchi,AM (L),GER,16.454483,2013-01-16 00:00:01,Bayer Leverkusen,Bundesliga,£16K - £170K,Temperamental,"Volatile, Media-friendly",Reasonable,Very Strong,12,11,1,1,3,10,13,5,13,18,2,7,7,10,5,17,13,...,8,11,12,2,2,7,13,6,1,4,8,4,12,6,1,2029-07-01 00:00:01,[AML],0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,r-2002111705,Ctr,Kadir Ok,"D (RL), WB/M/AM (R)",TUR,19.353867,2010-02-22 00:00:01,Bayer Leverkusen II,Regionalliga West,£30K - £60K,Balanced,Media-friendly,Strong,Very Strong,17,13,12,3,3,15,11,3,10,14,3,3,7,4,8,9,4,...,11,8,11,2,1,7,13,1,3,1,1,4,12,2,2,2029-07-01 00:00:01,"[DR, DL, WBR, MR, AMR]",0,0,0,0,1,1,1,1,0,0,0,0,0,1
3,93127195,Wnt,Odilon Kossounou,D (RC),CIV,28.487337,2001-01-04 00:00:01,Bayer Leverkusen,Bundesliga,£49.5M,Fairly Determined,Media-friendly,Reasonable,Very Strong,17,15,17,1,1,13,13,15,15,15,1,14,14,9,14,16,12,...,14,15,13,1,1,14,14,3,3,1,3,13,13,4,2,2029-07-01 00:00:01,"[DR, DC]",0,0,1,0,0,1,0,0,0,0,0,0,0,0
4,r-2002111746,Wnt,Julian Hutton-Woods,D (C),ENG,19.337440,2010-02-28 00:00:01,Bayer Leverkusen,Bundesliga,£475K - £2.2M,Driven,Level-headed,Reasonable,Very Strong,13,14,16,3,2,7,13,10,15,13,1,13,10,9,12,18,8,...,14,9,8,1,3,6,11,4,3,3,3,8,14,6,3,2029-07-01 00:00:01,[DC],0,0,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1740,2000104892,NaN,Eliesse Ben Seghir,"M (C), AM (RC)",FRA,24.369610,2005-02-16 00:00:01,Chelsea,Premier League,£109M - £156M,Resolute,Reserved,Fairly Strong,Very Strong,15,16,8,1,4,13,17,15,15,11,1,13,15,14,15,15,17,...,8,13,17,3,3,16,15,12,4,2,12,5,14,12,1,2029-07-01 00:00:01,"[MC, AMR, AMC]",1,0,0,0,1,0,0,0,1,0,0,0,0,0
1741,r-2002122517,Inj,Liam Van Den Bossche,"AM (RL), ST (C)",BEL,18.398357,2011-02-06 00:00:01,Young Reds,Nationale 1,£40K - £110K,Realist,"Outspoken, Volatile, Confrontational",Very Strong,Reasonable,13,15,9,3,1,8,13,6,8,4,4,5,8,12,8,15,13,...,6,5,10,2,2,6,7,9,2,4,7,4,12,3,1,2029-07-01 00:00:01,"[AMR, AML, ST]",0,0,0,1,1,0,0,0,0,1,0,0,0,0
1742,r-2002120802,Sct,Giorgos Konstantinou,"D (RL), WB (R)",CYP,18.026010,2011-06-22 00:00:01,Arsenal,Premier League,£14M - £20M,Light-Hearted,Unflappable,Reasonable,Very Strong,17,16,8,2,2,6,19,8,15,17,3,8,6,12,13,15,11,...,10,17,13,2,3,6,14,6,2,3,3,6,14,5,2,2029-07-01 00:00:01,"[DR, DL, WBR]",0,0,0,0,0,1,0,1,0,0,0,0,0,1
1743,r-2002069741,Sct,Matthias Weigel,D/WB (R),GER,21.056810,2008-06-10 00:00:01,FC Bayern II,3. Liga,£4.9M - £8.4M,Balanced,Media-friendly,Fairly Strong,Very Strong,13,12,6,1,2,6,14,14,17,16,3,14,12,12,14,11,8,...,14,17,15,3,1,13,16,8,1,2,4,5,18,6,3,2029-07-01 00:00:01,"[DR, WBR]",0,0,0,0,0,1,0,0,0,0,0,0,0,1


In [145]:
latest_export_date = df['Date'].max()
df['is_latest_export'] = df['Date'] == latest_export_date

df['GK'] = df['GK']*((5 * (df['Agi']+df['Ref']+ df['Cmd']+df['Cnt'])) + (3 * (df['1v1']+df['Ant']+df['Kic']+df['Pos'])) +(1 * (df['Acc']+df['Aer']+df['Dec']+df['Fir']+df['Han']+df['Pas']+df['TRO']+df['Vis'])))/40
df['DC'] = df['DC']*(((df['Acc']+ df['Pac']+df['Jum'])*5) +  (( df['Str'] + df['Cnt']+ df['Hea']+df['Dec'])*3) + (df['Det']+ df['Bra'] + df['Ant']+ df['Pos']))/31
df['DR'] = df['DR']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['Pos']+df['Bal']))/35
df['DL'] = df['DL']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['Pos']+df['Bal']))/35
df['MR'] = df['MR']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['AML'] = df['AML']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['DM'] = df['DM']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Pos'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['AMC'] = df['MC']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Vis'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['ST'] = df['ST']*(((df['Acc']+ df['Pac']+df['Jum'])*5) +  ((df['Dri']+ df['Ant'] + df['Cnt']+ df['Agi']+df['Bal'])*3) + (df['Det']+ df['Wor'] + df['Sta']+ df['Nat.1']))/34


In [146]:
all_ratings_df = df

In [147]:
positions_in_current_formation = ['GK', 'DC', 'DR', 'DL', 'DM', 'MR','AML', 'ST']

In [148]:
all_ratings_df['Positions'] = all_ratings_df.apply(lambda row: [col for col in positions_in_current_formation if row[col] > 0], axis=1)
all_ratings_df['Positions'] = all_ratings_df['Positions'].apply(lambda x: ', '.join(x))
all_ratings_df['Positions_filter'] = all_ratings_df['Positions']+', All'
team_df = all_ratings_df[all_ratings_df['Club'].str.contains(current_save, case=False, na=False)]


In [156]:
columns_to_calculate = ['GK', 'DC', 'DR', 'DL', 'DM', 'MR', 'AML', 'AMC', 'ST']

# Calculate the maximum value for each column in the selected subset
max_values = all_ratings_df[columns_to_calculate].max()

# Calculate the percentage of each value relative to the column maximum
percentage_df = all_ratings_df[columns_to_calculate] / max_values * 100

# Add percentage columns to the original DataFrame with new names
for col in columns_to_calculate:
    all_ratings_df[f'{col}_%'] = percentage_df[col].round(2)
    
all_ratings_df['Best Pos'] = percentage_df.idxmax(axis=1)  # Name of the column with the highest %
all_ratings_df['Best Pos Rating'] = percentage_df.max(axis=1).round(2)  # Value of the highest %

In [157]:
viz_df = all_ratings_df[['Name','Club','Age','Positions','Nat','Transfer Value','Positions_filter','GK_%','DC_%','DR_%','DL_%','DM_%','MR_%','ML_%','AML_%','ST_%','Best Pos','Best Pos Rating','is_latest_export']]
viz_df=viz_df.round(2)
viz_df = viz_df[viz_df['is_latest_export']==True]

In [158]:


app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Callback function to update the datatable
@app.callback(
    Output(component_id='my_squad', component_property='data'),
    [
        Input(component_id='age_range_slider', component_property='value'),
        Input(component_id='position_filter', component_property='value'),
        Input(component_id='is_latest_export_filter', component_property='value'),
        Input(component_id='club_filter', component_property='value')
    ]
)
def update_datatable(selected_age_range, selected_positions, is_latest_export, club_filter):
    # Make a copy of the global df to avoid modifying it directly
    df_filtering = viz_df.copy()
    df_filtering['Positions_filter'] = df_filtering['Positions_filter'].str.split(', ')

    # Filter the DataFrame based on the selected positions
    if 'All' in selected_positions:
        filtered_df = df_filtering
    else:
        filtered_df = df_filtering[df_filtering['Positions_filter'].apply(
            lambda x: any(pos in x for pos in selected_positions))]

    # Convert list in 'Positions' column to comma-separated strings for display
    filtered_df['Positions_filter'] = filtered_df['Positions_filter'].apply(lambda x: ', '.join(x))

    # Filter by 'is_latest_export'
   # filtered_df = filtered_df[filtered_df['is_latest_export'].isin(is_latest_export)]

    # Filter by age range
    filtered_df = filtered_df[
        (selected_age_range[0] <= filtered_df['Age']) &
        (filtered_df['Age'] <= selected_age_range[1])
    ]

    # Filter by club name
    if club_filter:  # Only filter if club_filter is not empty
        filtered_df = filtered_df[filtered_df['Club'].str.contains(club_filter, case=False, na=False)]


    # Return the filtered DataFrame as a list of dictionaries
    return filtered_df.to_dict('records')


# Define the layout of the app
app.layout = html.Div([
    dbc.Row(html.H1("Web App", style={'text-align': 'center'})),

    dbc.Row([
        dbc.Col(html.Div([
            dcc.RangeSlider(15, 40,
                            value=[15, 40],
                            id='age_range_slider'),
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="position_filter",
                         options=[{'label': pos, 'value': pos} for pos in ['GK', 'DC', 'DR', 'DL', 'DM','MR','AML', 'ST', 'All']],
                         multi=True,
                         value=['All']),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="is_latest_export_filter",
                         options=[{'label': 'True', 'value': True}, {'label': 'False', 'value': False}],
                         multi=True,
                         value=[True]),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Input(id="club_filter",
                      type='text',
                      placeholder='Enter a club name...',
                      debounce=True,
                      value=""),  # Default value
        ]), width=3),
    ], className='row'),

    html.Br(),
    dbc.Container(
        dbc.Row(
            dbc.Col(
                html.Div(
                    id='output_container',
                    children=[
                        dash_table.DataTable(
                            columns=[{"name": i, "id": i} for i in viz_df.columns],  # Include Positions column
                            data=viz_df.to_dict('records'),
                            page_size=15,
                            id='my_squad',
                            filter_action="native",
                            sort_action="native",
                            sort_mode="multi",
                            row_selectable="multi",
                            hidden_columns=[]  # Adjust hidden columns
                        )
                    ]
                ),
            ),
            className='row'
        ), fluid=True
    )
])




In [159]:
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [161]:

df= viz_df[viz_df['Club']=='Bayer Leverkusen']
formation_slots = {
    "GK": 1,
    "DR": 1,
    "DC": 2,
    "DL": 1,
    "MR": 1,
    "DM": 2,
    "AML": 1,
    "ST": 2,
}

# Add position-specific ratings to a single column for easier selection
position_columns = ["GK_%", "DC_%", "DR_%", "DL_%", "DM_%", "MR_%", "AML_%", "ST_%"]
position_map = {col[:-2]: col for col in position_columns}

# Create a melted DataFrame to handle player-position ratings easily
ratings = df.melt(
    id_vars=["Name", "Best Pos", "Best Pos Rating", "is_latest_export"],
    value_vars=position_columns,
    var_name="Position",
    value_name="Rating",
)

# Clean Position column to match "GK", "DC", etc.
ratings["Position"] = ratings["Position"].str[:-2]

# Sort by position and rating to prioritize the best fit
ratings = ratings.sort_values(by=["Position", "Rating"], ascending=[True, False])

# Function to assign players to the teams by selecting the highest-rated available player
def assign_players_to_team(formation_slots, ratings, available_players):
    team = {pos: [] for pos in formation_slots.keys()}
    # Iterate through each position and assign the highest-rated available player
    for pos, slots in formation_slots.items():
        # Select the highest-rated players for each position
        position_ratings = ratings[ratings["Position"] == pos]
        selected_players = []
        for _, row in position_ratings.iterrows():
            if row["Name"] in available_players and len(selected_players) < slots:
                selected_players.append((row["Name"], row["Rating"]))
                available_players.remove(row["Name"])
        
        team[pos] = selected_players

    return team

# Available players list for both teams (starts as all players)
available_players_for_team1 = set(df["Name"])

# Assign players to Team 1
team1 = assign_players_to_team(formation_slots, ratings, available_players_for_team1)

# Available players list for Team 2 (only those who weren't selected for Team 1)
available_players_for_team2 = set(df["Name"]) - set(player[0] for players in team1.values() for player in players)

# Assign players to Team 2 using remaining players
team2 = assign_players_to_team(formation_slots, ratings, available_players_for_team2)

# Display Team 1 and Team 2 in the desired format
def display_team(team, team_name):
    print(f"{team_name}:")
    position_order = ["GK", "DR", "DC", "DC", "DL", "MR", "DM", "DM", "AML", "ST", "ST"]
    
    for pos in position_order:
        if pos in team:
            for player in team[pos]:
                print(f"{pos}: {player[0]} {player[1]:.2f}")
        else:
            print(f"{pos}: None")  # If no player is assigned to the position

    print("\n")

# Display both teams
display_team(team1, "Team 1")
display_team(team2, "Team 2")

Team 1:
GK: Matěj Kovář 85.69
DR: Matías González 93.90
DC: Grégory Soler 95.62
DC: Jordan Millbank 92.11
DC: Grégory Soler 95.62
DC: Jordan Millbank 92.11
DL: Piero Hincapié 83.79
MR: Lamine Yamal 91.88
DM: Mika Faye 77.85
DM: Tom Krauß 73.94
DM: Mika Faye 77.85
DM: Tom Krauß 73.94
AML: Archie Brown 84.39
ST: Junior Kroupi 83.19
ST: Simon Böhm 59.93
ST: Junior Kroupi 83.19
ST: Simon Böhm 59.93


Team 2:
GK: Franco Maffei 85.69
DR: Jeremie Frimpong 87.32
DC: Odilon Kossounou 90.19
DC: António Silva 85.29
DC: Odilon Kossounou 90.19
DC: António Silva 85.29
DL: Paul Morera 68.13
MR: Festy Ebosele 79.23
DM: Edmond Tapsoba 69.25
DM: Onat Acı 53.81
DM: Edmond Tapsoba 69.25
DM: Onat Acı 53.81
AML: Scott McLaren 78.25
ST: Vladimir Galiev 48.76
ST: Ingo Dann 47.56
ST: Vladimir Galiev 48.76
ST: Ingo Dann 47.56




In [155]:
print(df)

                   Name              Club    Age    Positions  Nat  \
0             Ady Heale  Bayer Leverkusen  15.84               ENG   
1         Gaku Ideguchi  Bayer Leverkusen  16.45          AML  GER   
3      Odilon Kossounou  Bayer Leverkusen  28.49       DC, DR  CIV   
4   Julian Hutton-Woods  Bayer Leverkusen  19.34           DC  ENG   
5         Festy Ebosele  Bayer Leverkusen  26.91       DR, MR  IRL   
6         Grégory Soler  Bayer Leverkusen  21.01       DC, DR  BEL   
7         Junior Kroupi  Bayer Leverkusen  23.02           ST  FRA   
8             Mika Faye  Bayer Leverkusen  24.96   DC, DL, DM  SEN   
9       Jaromír Valenta  Bayer Leverkusen  19.23           GK  CZE   
10       Simone Pivetta  Bayer Leverkusen  18.56   DC, DR, DM  ITA   
11             Onat Acı  Bayer Leverkusen  18.12       DC, DM  TUR   
12         Lamine Yamal  Bayer Leverkusen  21.97  MR, AML, ST  ESP   
13      Vladimir Galiev  Bayer Leverkusen  18.04       MR, ST  GER   
14       Edmond Taps